In [1]:
import torch

from constants import DATA_FILE_BREAST_CANCER, COLUMN_NAMES, FEATURE_NAMES, LABEL_NAME, SEED
import net
from utils import read_data_to_df, create_sparsity
from net import (
    DNN, BreastCancerDataset,
    train, test
)
import torch.optim as optim
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
# test the tree embedding function and gbr model on train test split
from utils import read_data_to_df, create_sparsity
from tree_embeddings import create_tree_embeddings

# test imports
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb
from utils import set_seed
from net import train_kfold as train_kfold_dnn
from TN_net import TN_DNN, train_kfold as train_kfold_tn
from sklearn.model_selection import StratifiedKFold
import xgboost as xgb
import pandas as pd


ModuleNotFoundError: No module named 'sklearn'

In [ ]:
df = read_data_to_df(DATA_FILE_BREAST_CANCER, COLUMN_NAMES)

X = df[FEATURE_NAMES].values
y = df[LABEL_NAME].values

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)

#describe the data
print("Training set:")
print(f"  Features shape: {X_train.shape}")
print(f"  Labels shape: {y_train.shape}")
print("Test set:")
print(f"  Features shape: {X_test.shape}")
print(f"  Labels shape: {y_test.shape}")
# Check for class imbalance
unique, counts = np.unique(y_train, return_counts=True)
print("Class distribution in training set:")
for cls, count in zip(unique, counts):
    print(f"  Class {cls}: {count} samples ({count / len(y_train) * 100:.2f}%)")

print("Class distribution in test set:")
unique, counts = np.unique(y_test, return_counts=True)
for cls, count in zip(unique, counts):
    print(f"  Class {cls}: {count} samples ({count / len(y_test) * 100:.2f}%)")


Training set:
  Features shape: (455, 30)
  Labels shape: (455,)
Test set:
  Features shape: (114, 30)
  Labels shape: (114,)
Class distribution in training set:
  Class B: 286 samples (62.86%)
  Class M: 169 samples (37.14%)
Class distribution in test set:
  Class B: 71 samples (62.28%)
  Class M: 43 samples (37.72%)


In [ ]:

df = read_data_to_df(DATA_FILE_BREAST_CANCER, COLUMN_NAMES)
df_sparse = create_sparsity(df, FEATURE_NAMES, sparsity_level=0.4, random_state=SEED)

df_train, df_test = train_test_split(df_sparse, test_size=0.2, random_state=SEED)

gbr, embeddings = create_tree_embeddings(df_train)

# Use DMatrix for predictions with XGBoost
dtest = xgb.DMatrix(df_test[FEATURE_NAMES].values)
gbr_predictions = gbr.predict(dtest)

# Encode labels: B (benign) = 0, M (malignant) = 1
y_test_encoded = np.where(df_test[LABEL_NAME].values == 'M', 1, 0)

gbr_auc = roc_auc_score(y_test_encoded, gbr_predictions)
gbr_ap = average_precision_score(y_test_encoded, gbr_predictions)
print(f"Gradient Boosting Classifier - AUC: {gbr_auc:.4f}, AP: {gbr_ap:.4f}")


Gradient Boosting Classifier - AUC: 0.9669, AP: 0.9684


In [7]:
set_seed(SEED)

df = read_data_to_df(DATA_FILE_BREAST_CANCER, COLUMN_NAMES)
df_sparse_40 = create_sparsity(df, FEATURE_NAMES, sparsity_level=0.4, random_state=SEED)

# Select the dataset to use: df, df_sparse_20, or df_sparse_40
df_to_use = df_sparse_40
df_to_use[FEATURE_NAMES] = df_to_use[FEATURE_NAMES].apply(pd.to_numeric, errors="coerce")

def evaluate_gbdt_kfold(df, n_splits=5, num_trees=80, max_depth=9, verbose=True):
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    labels = df[LABEL_NAME].values

    fold_aurocs = []
    fold_auprcs = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(df, labels), start=1):
        if verbose:
            print(f"\n{'='*50}")
            print(f"  Fold {fold}/{n_splits}")
            print(f"{'='*50}")

        fold_train_df = df.iloc[train_idx].copy()
        fold_val_df = df.iloc[val_idx].copy()

        X_train = fold_train_df[FEATURE_NAMES].values
        y_train = np.where(fold_train_df[LABEL_NAME].values == "M", 1, 0)
        dtrain = xgb.DMatrix(X_train, label=y_train)

        params = {
            "n_estimators": num_trees,
            "max_depth": max_depth,
            "objective": "reg:squarederror",
            "verbosity": 0,
            "seed": SEED,
            "nthread": 1,
        }
        model = xgb.train(params, dtrain, num_boost_round=num_trees)

        X_val = fold_val_df[FEATURE_NAMES].values
        y_val = np.where(fold_val_df[LABEL_NAME].values == "M", 1, 0)
        dval = xgb.DMatrix(X_val)
        preds = model.predict(dval)

        auroc = roc_auc_score(y_val, preds)
        auprc = average_precision_score(y_val, preds)
        fold_aurocs.append(auroc)
        fold_auprcs.append(auprc)

        if verbose:
            print(f"Fold {fold} — AUROC: {auroc:.4f}  AUPRC: {auprc:.4f}")

    mean_auroc = np.mean(fold_aurocs)
    std_auroc = np.std(fold_aurocs)
    mean_auprc = np.mean(fold_auprcs)
    std_auprc = np.std(fold_auprcs)

    if verbose:
        print(f"\n{'='*50}")
        print(f"  Cross-Validation Results ({n_splits} folds)")
        print(f"  AUROC: {mean_auroc:.4f} ± {std_auroc:.4f}")
        print(f"  AUPRC: {mean_auprc:.4f} ± {std_auprc:.4f}")
        print(f"{'='*50}")

    return {
        "fold_aurocs": fold_aurocs,
        "fold_auprcs": fold_auprcs,
        "mean_auroc": mean_auroc,
        "std_auroc": std_auroc,
        "mean_auprc": mean_auprc,
        "std_auprc": std_auprc,
    }

cv_results_gbdt = evaluate_gbdt_kfold(
    df_to_use,
    n_splits=5,
    num_trees=80,
    max_depth=9,
    verbose=False
 )
print(f"GBDT CV AUROC: {cv_results_gbdt['mean_auroc']:.4f} +/- {cv_results_gbdt['std_auroc']:.4f}")
print(f"GBDT CV AUPRC: {cv_results_gbdt['mean_auprc']:.4f} +/- {cv_results_gbdt['std_auprc']:.4f}")

cv_results_dnn = train_kfold_dnn(
    df_to_use,
    DNN,
    n_splits=5,
    num_epochs=25,
    lr=1e-3,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=False
 )
print(f"DNN CV AUROC: {cv_results_dnn['mean_auroc']:.4f} +/- {cv_results_dnn['std_auroc']:.4f}")
print(f"DNN CV AUPRC: {cv_results_dnn['mean_auprc']:.4f} +/- {cv_results_dnn['std_auprc']:.4f}")

cv_results_tn = train_kfold_tn(
    df_to_use,
    TN_DNN,
    n_splits=5,
    num_epochs=25,
    lr=1e-3,
    device="cuda" if torch.cuda.is_available() else "cpu",
    verbose=False

 )
print(f"TN-DNN CV AUROC: {cv_results_tn['mean_auroc']:.4f} +/- {cv_results_tn['std_auroc']:.4f}")
print(f"TN-DNN CV AUPRC: {cv_results_tn['mean_auprc']:.4f} +/- {cv_results_tn['std_auprc']:.4f}")

GBDT CV AUROC: 0.9762 +/- 0.0118
GBDT CV AUPRC: 0.9712 +/- 0.0088
DNN CV AUROC: 0.8324 +/- 0.0413
DNN CV AUPRC: 0.7828 +/- 0.0511
TN-DNN CV AUROC: 0.9811 +/- 0.0072
TN-DNN CV AUPRC: 0.9755 +/- 0.0074
